# exp_002 — Quantization with llama.cpp GGUF

Load measured summaries and a resolved artifact manifest. The notebook does not run the benchmark or invent missing measurements.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = next(
    candidate for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / 'src').is_dir() and (candidate / 'experiments').is_dir()
)
sys.path.insert(0, str(ROOT / 'src'))
from llm_lab.analysis.quantization import recommend_baseline, tradeoff_rows
from llm_lab.quantization import QuantizationManifest

RESULTS_DIR = ROOT / 'experiments/exp_002-quantization_llama_cpp_gguf/results'
SUMMARY_PATH = RESULTS_DIR / 'processed/summary.csv'
MANIFEST_PATH = RESULTS_DIR / 'manifest.json'
if not SUMMARY_PATH.is_file():
    raise FileNotFoundError(f'measured summary is required: {SUMMARY_PATH}')
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f'resolved manifest is required: {MANIFEST_PATH}')

manifest = QuantizationManifest.from_record(json.loads(MANIFEST_PATH.read_text()))
summaries = pd.read_csv(SUMMARY_PATH)
required_columns = {
    'condition_id', 'attempted_n', 'scored_n', 'correct_n', 'failure_n',
    'scored_accuracy', 'end_to_end_success', 'failure_rate',
    'median_stream_ttft_s',
    'median_prompt_throughput_proxy_tok_s',
    'median_post_first_chunk_output_tok_s',
    'median_peak_memory_bytes',
}
missing_columns = required_columns - set(summaries.columns)
if missing_columns:
    raise ValueError(f'summary is missing required columns: {sorted(missing_columns)}')

rows = tradeoff_rows(summaries.to_dict('records'), manifest)
frame = pd.DataFrame(rows)
frame

In [ ]:
def accuracy_vs_memory(frame: pd.DataFrame):
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(frame['median_peak_memory_bytes'] / 1e9, frame['end_to_end_success'])
    for _, row in frame.iterrows():
        ax.annotate(row['label'], (row['median_peak_memory_bytes'] / 1e9, row['end_to_end_success']))
    ax.set_xlabel('Median peak memory (GB)')
    ax.set_ylabel('End-to-end success (correct / attempted)')
    ax.set_title('End-to-end success vs peak memory')
    ax.set_ylim(0, 1.05)
    return fig

RESULTS_DIR.joinpath('figures').mkdir(parents=True, exist_ok=True)
accuracy_figure = accuracy_vs_memory(frame)
accuracy_figure.savefig(RESULTS_DIR / 'figures' / 'accuracy-vs-memory.png', dpi=160, bbox_inches='tight')
accuracy_figure

In [ ]:
def speed_vs_memory(frame: pd.DataFrame):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True)
    x = frame['median_peak_memory_bytes'] / 1e9
    axes[0].scatter(x, frame['median_prompt_throughput_proxy_tok_s'])
    axes[1].scatter(x, frame['median_post_first_chunk_output_tok_s'])
    for axis, column in zip(axes, ('median_prompt_throughput_proxy_tok_s', 'median_post_first_chunk_output_tok_s')):
        for _, row in frame.iterrows():
            axis.annotate(row['label'], (row['median_peak_memory_bytes'] / 1e9, row[column]))
        axis.set_xlabel('Median peak memory (GB)')
        axis.set_ylabel('tokens / second')
    axes[0].set_title('Prompt-throughput proxy vs memory')
    axes[1].set_title('Post-first-chunk output throughput vs memory')
    return fig

speed_figure = speed_vs_memory(frame)
speed_figure.savefig(RESULTS_DIR / 'figures' / 'speed-vs-memory.png', dpi=160, bbox_inches='tight')
speed_figure

In [ ]:
recommendation = recommend_baseline(rows, accuracy_tolerance=0.02)
print({
    'recommended_condition': recommendation['condition_id'],
    'recommended_label': recommendation['label'],
    'scored_accuracy': recommendation['scored_accuracy'],
    'end_to_end_success': recommendation['end_to_end_success'],
    'failure_rate': recommendation['failure_rate'],
    'artifact_size_bytes': recommendation['artifact_size_bytes'],
    'best_end_to_end_success': recommendation['best_end_to_end_success'],
    'accuracy_tolerance': recommendation['accuracy_tolerance'],
})